In [2]:
import pandas as pd
import numpy as np

In [3]:
riders = pd.read_csv("../dataset/processed/riders_features.csv")

In [4]:
print(riders.shape)
riders.head()

(45493, 33)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Day_of_Week,Month,Weekend,Peak_Period,Trip_Distance_km,Traffic_Score,Weather_Score,Vehicle_Score,Rider_Experience,Workload
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,2022-02-12,21:55,...,Saturday,2,1,Dinner,10.280582,4,4,4,151.2,3.0
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,2022-02-13,14:55,...,Sunday,2,1,Lunch,6.242319,3,5,4,98.7,1.0
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,2022-03-04,17:30,...,Friday,3,0,Normal,13.787860,2,6,3,108.1,1.0
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,2022-02-13,09:20,...,Sunday,2,1,Breakfast,2.930258,1,6,4,146.2,0.0
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,2022-02-14,19:50,...,Monday,2,0,Dinner,19.396618,4,4,3,112.8,1.0


In [5]:
required_cols = [
    "Traffic_Score",
    "Weather_Score",
    "Trip_Distance_km",
    "multiple_deliveries",
    "Time_taken (min)"
]

missing = [c for c in required_cols if c not in riders.columns]

print("Missing columns:", missing)

Missing columns: []


In [6]:
print(riders.columns.tolist())

['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weather_conditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken (min)', 'Order_Time', 'Order_Hour', 'Pickup_Hour', 'Day_of_Week', 'Month', 'Weekend', 'Peak_Period', 'Trip_Distance_km', 'Traffic_Score', 'Weather_Score', 'Vehicle_Score', 'Rider_Experience', 'Workload']


In [7]:

# Normalize features
riders["traffic_norm"] = (
    riders["Traffic_Score"] - riders["Traffic_Score"].min()
) / (
    riders["Traffic_Score"].max() - riders["Traffic_Score"].min()
)

riders["weather_norm"] = (
    riders["Weather_Score"] - riders["Weather_Score"].min()
) / (
    riders["Weather_Score"].max() - riders["Weather_Score"].min()
)

riders["distance_norm"] = (
    riders["Trip_Distance_km"] - riders["Trip_Distance_km"].min()
) / (
    riders["Trip_Distance_km"].max() - riders["Trip_Distance_km"].min()
)

riders["workload_norm"] = (
    riders["Workload"] - riders["Workload"].min()
) / (
    riders["Workload"].max() - riders["Workload"].min()
)

In [8]:
# Operational stage scores
o2a_score = 0.4 * riders["traffic_norm"] + 0.6 * riders["workload_norm"]

fm_score = 0.7 * riders["distance_norm"] + 0.3 * riders["traffic_norm"]

wt_score = 0.7 * riders["weather_norm"] + 0.3 * riders["workload_norm"]

lm_score = 0.6 * riders["distance_norm"] + 0.4 * riders["traffic_norm"]

In [9]:
scores = (
    o2a_score +
    fm_score +
    wt_score +
    lm_score
)

riders["O2A_Ratio"] = o2a_score / scores
riders["FM_Ratio"] = fm_score / scores
riders["WT_Ratio"] = wt_score / scores
riders["LM_Ratio"] = lm_score / scores

In [10]:
eta = riders["Time_taken (min)"]

riders["O2A_Target"] = eta * riders["O2A_Ratio"]
riders["FM_Target"] = eta * riders["FM_Ratio"]
riders["WT_Target"] = eta * riders["WT_Ratio"]
riders["LM_Target"] = eta * riders["LM_Ratio"]

In [11]:
check = (
    riders["O2A_Target"] +
    riders["FM_Target"] +
    riders["WT_Target"] +
    riders["LM_Target"]
)

print("Maximum Error:", np.abs(check - eta).max())

Maximum Error: 1.4210854715202004e-14


In [12]:
riders.to_csv(
    "../dataset/processed/riders_multistage.csv",
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.
